# Estabilidade

Uma rede pode ter a arquitetura certa, a perda certa e o otimizador certo, e ainda assim não treinar. A cada camada o sinal que atravessa a rede é multiplicado pelos pesos, e o gradiente que volta faz o mesmo. Quando ele chega quase nulo às primeiras camadas, elas param de aprender; quando chega enorme, um único passo destrói os pesos.

Este material usa a mesma MLP do material anterior no MNIST e aplica, uma de cada vez, as quatro ferramentas que o PyTorch oferece para manter essas escalas sob controle: a inicialização dos pesos, a batch normalization, o gradient clipping e o learning rate scheduling.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

In [ ]:
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Dados

O MNIST entra normalizado, como nos materiais anteriores. Os valores 0.1307 e 0.3081 são a média e o desvio padrão dos pixels no conjunto de treino, e a transformação $x' = (x - \mu) / \sigma$ deixa a entrada com média zero e desvio padrão um. É a primeira escala a controlar: se a entrada já chega grande ou pequena demais, tudo o que vem depois herda o problema.

Para que cada treinamento leve poucos segundos, usa-se um subconjunto de 3.000 exemplos de treino e 1.000 de validação.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.1307,), std=(0.3081,)),
])

full_train_set = datasets.MNIST(root="data", train=True, download=True, transform=transform)
full_test_set = datasets.MNIST(root="data", train=False, download=True, transform=transform)

In [ ]:
train_set = Subset(full_train_set, range(3000))
validation_set = Subset(full_test_set, range(1000))

train_dataloader = DataLoader(train_set, batch_size=64, shuffle=True)
validation_dataloader = DataLoader(validation_set, batch_size=500, shuffle=False)

print(f"treino: {len(train_set)}, validação: {len(validation_set)}")

## O modelo

A rede é a MLP de sempre: três camadas ocultas com ReLU e uma saída de dez logits. Dois argumentos deixam o modelo pronto para as seções seguintes. O primeiro escolhe a função de ativação, e o segundo insere uma `nn.BatchNorm1d` depois de cada `nn.Linear`, opção que fica desligada até a seção de batch normalization.

In [ ]:
class MLP(nn.Module):
    def __init__(self, activation=nn.ReLU, batch_norm=False):
        super().__init__()
        sizes = [28 * 28, 256, 128, 64]
        layers = [nn.Flatten()]   # [batch, 1, 28, 28] -> [batch, 784]

        for in_features, out_features in zip(sizes, sizes[1:]):
            layers.append(nn.Linear(in_features, out_features))
            if batch_norm:
                layers.append(nn.BatchNorm1d(out_features))
            layers.append(activation())

        layers.append(nn.Linear(sizes[-1], 10))
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)   # [batch, 10]

As funções de treino e avaliação são as dos materiais anteriores. A de treino ganhou dois argumentos opcionais, `scheduler` e `max_norm`, que ficam em `None` até as seções que os usam, e devolve a acurácia de validação de cada época.

In [ ]:
criterion = nn.CrossEntropyLoss()


def evaluate(model, dataloader):
    model.eval()
    correct = 0

    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(dim=1) == labels).sum().item()

    return correct / len(dataloader.dataset)

In [ ]:
def train(model, optimizer, scheduler=None, max_norm=None, epochs=15):
    history = {"loss": [], "accuracy": [], "lr": []}

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0

        for images, labels in train_dataloader:
            images, labels = images.to(device), labels.to(device)
            loss = criterion(model(images), labels)

            optimizer.zero_grad()
            loss.backward()

            if max_norm is not None:
                nn.utils.clip_grad_norm_(model.parameters(), max_norm)

            optimizer.step()
            running_loss += loss.item() * images.size(0)

        history["lr"].append(optimizer.param_groups[0]["lr"])

        if scheduler is not None:
            scheduler.step()

        history["loss"].append(running_loss / len(train_set))
        history["accuracy"].append(evaluate(model, validation_dataloader))

    return history

Cada treinamento acrescenta o seu histórico ao dicionário `histories`, e `plot_history` desenha todos os que já existem, para que cada versão da rede seja comparada com as anteriores. A perda de treino fica em escala logarítmica, porque as versões desta aula diferem em ordens de grandeza.

In [ ]:
histories = {}


def plot_history(histories):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    for name, history in histories.items():
        ax1.plot(history["loss"], marker="o", label=name)
        ax2.plot(history["accuracy"], marker="o", label=name)

    ax1.set_yscale("log")
    ax1.set_xlabel("época")
    ax1.set_ylabel("entropia cruzada de treino")
    ax1.legend()
    ax1.grid(True)

    ax2.set_ylim(0, 1)
    ax2.set_xlabel("época")
    ax2.set_ylabel("acurácia de validação")
    ax2.legend()
    ax2.grid(True)
    plt.show()

## Vanishing e exploding gradients

Ao atravessar uma camada de trás para frente, o gradiente é multiplicado pela derivada da ativação e pelos pesos daquela camada. O gradiente que chega à primeira camada passou por uma sequência dessas multiplicações, e o produto muda exponencialmente com a profundidade: se os fatores são menores que um, ele encolhe até desaparecer, e é o **vanishing gradient**; se são maiores que um, ele cresce sem controle, e é o **exploding gradient**.

A derivada da sigmoid vale no máximo $0.25$, e quase zero nas pontas, então ela puxa o produto para baixo em toda camada. A da ReLU vale exatamente um na parte positiva, e por isso ela é a ativação padrão em redes profundas.

Para ver isso no PyTorch, basta um `backward` em um lote e a norma do gradiente de cada camada, que fica em `layer.weight.grad`.

In [ ]:
images, labels = next(iter(train_dataloader))
images, labels = images.to(device), labels.to(device)


def gradient_norms(model):
    loss = criterion(model(images), labels)

    model.zero_grad()
    loss.backward()

    return [layer.weight.grad.norm().item() for layer in model.layers if isinstance(layer, nn.Linear)]

In [ ]:
torch.manual_seed(0)
sigmoid_norms = gradient_norms(MLP(nn.Sigmoid).to(device))

torch.manual_seed(0)
relu_norms = gradient_norms(MLP(nn.ReLU).to(device))

plt.figure(figsize=(8, 5))
plt.plot(sigmoid_norms, marker="o", label="sigmoid")
plt.plot(relu_norms, marker="o", label="ReLU")
plt.yscale("log")
plt.xlabel("camada")
plt.ylabel("norma do gradiente")
plt.legend()
plt.grid(True)
plt.show()

Com a sigmoid, a norma da primeira camada é quase quarenta vezes menor que a da última, e as camadas iniciais quase não são atualizadas. Com a ReLU as quatro normas ficam na mesma ordem de grandeza. Treinar as duas redes mostra o que isso significa na prática.

In [ ]:
torch.manual_seed(42)
sigmoid_model = MLP(nn.Sigmoid).to(device)
histories["sigmoid"] = train(sigmoid_model, torch.optim.SGD(sigmoid_model.parameters(), lr=0.1))

torch.manual_seed(42)
relu_model = MLP(nn.ReLU).to(device)
histories["ReLU"] = train(relu_model, torch.optim.SGD(relu_model.parameters(), lr=0.1))

plot_history(histories)

A rede com sigmoid fica perto dos 10% de acurácia, que é o acaso entre dez classes, enquanto a mesma rede com ReLU passa dos 90%. Trocar a ativação já resolve boa parte do problema, e as próximas seções cuidam do que sobra.

## Inicialização dos pesos

Antes da primeira iteração os pesos precisam de algum valor. A escolha mais ingênua é zerar tudo, e vale ver o que acontece com o gradiente nesse caso. A `model.apply` percorre todos os submódulos e aplica a função recebida, que é o jeito idiomático de inicializar uma rede no PyTorch.

In [ ]:
def zero_init(module):
    if isinstance(module, nn.Linear):
        nn.init.zeros_(module.weight)
        nn.init.zeros_(module.bias)


torch.manual_seed(0)
zero_model = MLP(nn.Sigmoid).to(device)
zero_model.apply(zero_init)

print([f"{norm:.4f}" for norm in gradient_norms(zero_model)])
print(zero_model.layers[-1].weight.grad[:3, :4])

Só a última camada recebe gradiente. O gradiente que volta para as anteriores é multiplicado pelos pesos, que são zero, e nada chega. E mesmo a última não aprende nada útil: todas as suas entradas são iguais, já que cada unidade oculta calcula $\sigma(0) = 0.5$, e por isso as colunas do gradiente são idênticas. Depois do passo, as unidades de cada camada continuam iguais entre si, e nenhuma quantidade de treinamento quebra essa **simetria**. Inicializar com qualquer outra constante tem o mesmo problema.

A saída é sortear os pesos, mas a escala do sorteio decide se o sinal atravessa a rede. Se a variância é pequena demais, as ativações encolhem camada a camada; se é grande demais, elas explodem. As duas escolhas usuais ajustam a variância ao tamanho da camada,

$$
\text{Xavier (Glorot):} \quad \sigma^2 = \frac{2}{n_\text{in} + n_\text{out}}
\qquad
\text{He (Kaiming):} \quad \sigma^2 = \frac{2}{n_\text{in}}
$$

em que $n_\text{in}$ e $n_\text{out}$ são o número de entradas e de saídas da camada. A de Xavier, ou Glorot, supõe uma ativação simétrica em torno de zero, como a tanh ou a sigmoid; a de He, ou Kaiming, tem o fator dobrado para compensar a metade que a ReLU zera, e é a que se usa com ela. No PyTorch as duas estão em `nn.init`, junto com as demais.

In [ ]:
def he_init(module):
    if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
        nn.init.zeros_(module.bias)


torch.manual_seed(42)
he_model = MLP(nn.ReLU).to(device)
he_model.apply(he_init)

histories["ReLU + He"] = train(he_model, torch.optim.SGD(he_model.parameters(), lr=0.1))
plot_history(histories)

A acurácia final é parecida com a da inicialização padrão, mas a primeira época já começa acima de 80%, contra menos de 60% antes. A inicialização padrão do `nn.Linear` é mais conservadora que a de He, o que custa algumas épocas aqui e impede o treinamento por completo em redes mais profundas.

## Batch normalization

A inicialização acerta a escala das ativações no instante zero, mas os pesos mudam a cada passo, e a escala muda com eles. A **batch normalization** recalibra essa escala em toda passada, normalizando cada ativação dentro do mini lote,

$$
\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}
\qquad
y_i = \gamma \hat{x}_i + \beta
$$

em que $\mu_B$ e $\sigma_B^2$ são a média e a variância do lote, $\epsilon$ evita divisão por zero, e $\gamma$ e $\beta$ são parâmetros treináveis que devolvem à rede a liberdade de escolher a escala e o deslocamento.

A camada é a `nn.BatchNorm1d` para vetores e a `nn.BatchNorm2d` para imagens, e o argumento é o número de unidades da camada anterior. Ela entra entre a `nn.Linear` e a ativação, que é o que o argumento `batch_norm` do modelo faz.

In [ ]:
batch_norm = nn.BatchNorm1d(4)
x = torch.randn(32, 4) * 10 + 5

print(f"entrada: média {x.mean():.2f}, desvio {x.std():.2f}")
print(f"saída:   média {batch_norm(x).mean():.2f}, desvio {batch_norm(x).std():.2f}")

A escala das ativações é também o que decide qual taxa de aprendizado a rede aguenta: um passo grande demais leva os pesos para uma região onde a perda é enorme, o gradiente seguinte é enorme, e o próximo passo é pior ainda. Com `lr=0.5`, cinco vezes a taxa usada até aqui, a rede sem normalização diverge; com a camada, a mesma taxa treina.

In [ ]:
torch.manual_seed(42)
diverged_model = MLP(nn.ReLU).to(device)
diverged_model.apply(he_init)

histories["ReLU + He, lr 0.5"] = train(diverged_model, torch.optim.SGD(diverged_model.parameters(), lr=0.5))
print(f"perda final sem batch norm: {criterion(diverged_model(images), labels).item()}")

In [ ]:
torch.manual_seed(42)
batch_norm_model = MLP(nn.ReLU, batch_norm=True).to(device)
batch_norm_model.apply(he_init)

histories["ReLU + He + batch norm, lr 0.5"] = train(batch_norm_model, torch.optim.SGD(batch_norm_model.parameters(), lr=0.5))
plot_history(histories)

Sem a normalização a perda vira `nan`, e a acurácia congela em 8,5%, que é a fração de zeros na validação: com todos os logits `nan`, o `argmax` devolve sempre a primeira classe. Depois que os pesos viram `nan` não há volta, porque todo gradiente calculado a partir deles também é `nan`. Com a camada, a mesma taxa chega a 93%, a melhor curva até aqui. Essa é a vantagem prática da batch normalization: como ela repõe a escala das ativações a cada passada, taxas de aprendizado maiores deixam de ser perigosas, e o treinamento fica menos sensível à inicialização.

A camada tem comportamentos diferentes no treino e na avaliação. Durante o treino ela usa as estatísticas do lote atual e vai acumulando uma média móvel delas; na avaliação usa a média acumulada, para que a previsão de um exemplo não dependa dos outros exemplos do lote. É o `model.train()` e o `model.eval()` que alternam entre os dois modos, e as funções `train` e `evaluate` já fazem essa troca. Esquecer o `eval()` é o erro mais comum com essa camada: um único exemplo em modo de treino nem chega a passar, porque não há como calcular a variância de um elemento só.

## Gradient clipping

A batch normalization evitou a divergência normalizando as ativações. O **gradient clipping** ataca o mesmo problema pelo outro lado, limitando o tamanho do passo em vez da escala do sinal. Se a norma de todos os gradientes juntos passa de um teto, todos são multiplicados pelo mesmo fator até que ela fique igual ao teto,

$$
g \leftarrow g \cdot \min\left(1, \frac{c}{\|g\|}\right)
$$

em que $c$ é o valor máximo permitido. A direção do passo não muda, só o seu tamanho. No PyTorch é a `nn.utils.clip_grad_norm_`, chamada entre o `backward` e o `step`, como já está na função `train`. Ela devolve a norma antes do recorte, que é um bom valor para acompanhar durante o treinamento.

In [ ]:
torch.manual_seed(0)
large_model = MLP(nn.ReLU).to(device)
with torch.no_grad():
    for layer in large_model.layers:
        if isinstance(layer, nn.Linear):
            layer.weight.mul_(5)

gradient_norms(large_model)
norm_before = nn.utils.clip_grad_norm_(large_model.parameters(), max_norm=1.0)
norm_after = torch.cat([p.grad.flatten() for p in large_model.parameters()]).norm()

print(f"norma antes: {norm_before:.2f}, depois: {norm_after:.2f}")

In [ ]:
torch.manual_seed(42)
clipped_model = MLP(nn.ReLU).to(device)
clipped_model.apply(he_init)

histories["ReLU + He + clipping, lr 0.5"] = train(clipped_model, torch.optim.SGD(clipped_model.parameters(), lr=0.5), max_norm=1.0)
plot_history(histories)

A rede que divergia com `lr=0.5` agora treina, sem normalização nenhuma, e chega perto da versão com batch normalization. O clipping não escolhe uma boa taxa de aprendizado, só impede que um passo isolado destrua os pesos, o que o torna útil onde a normalização por lote não se aplica bem: é o caso das redes recorrentes e dos transformers, em que ele é praticamente obrigatório.

## Learning rate scheduling

Um passo grande ajuda no começo, quando os parâmetros estão longe de qualquer mínimo, e atrapalha no fim, quando falta apenas ajustar. Um **scheduler** reduz a taxa de aprendizado ao longo do treinamento. O `StepLR` a multiplica por um fator fixo a cada tantas épocas; o `CosineAnnealingLR` a leva suavemente de $\eta_0$ até zero,

$$
\eta_t = \frac{\eta_0}{2} \left(1 + \cos \frac{\pi t}{T}\right)
$$

em que $t$ é a época e $T$ é o total de épocas. O scheduler recebe o otimizador e passa a controlar o `lr` dele; o seu `step` é chamado uma vez por época, depois do laço dos lotes, como já está na função `train`.

A comparação usa a rede com batch normalization da seção anterior e parte de `lr=2.0`, quatro vezes a taxa que fazia a rede divergir sem a camada. É com uma taxa inicial agressiva assim que o efeito do scheduling aparece.

In [ ]:
schedulers = {
    "constante": lambda optimizer: None,
    "StepLR": lambda optimizer: torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1),
    "cosine": lambda optimizer: torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15),
}

scheduled = {}

for name, make_scheduler in schedulers.items():
    torch.manual_seed(42)
    model = MLP(nn.ReLU, batch_norm=True).to(device)
    model.apply(he_init)

    optimizer = torch.optim.SGD(model.parameters(), lr=2.0)
    scheduled[name] = train(model, optimizer, scheduler=make_scheduler(optimizer))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for name, history in scheduled.items():
    ax1.plot(history["lr"], marker="o", label=name)
    ax2.plot(history["loss"], marker="o", label=name)

ax1.set_xlabel("época")
ax1.set_ylabel("taxa de aprendizado")
ax1.legend()
ax1.grid(True)

ax2.set_yscale("log")
ax2.set_xlabel("época")
ax2.set_ylabel("entropia cruzada de treino")
ax2.legend()
ax2.grid(True)
plt.show()

O gráfico da esquerda é o agendamento em si: o `StepLR` desce em degraus, dividindo a taxa por dez a cada cinco épocas, e o `CosineAnnealingLR` desce continuamente de 2.0 até quase zero.

O da direita mostra o efeito na perda de treino. Com a taxa constante, a perda para de cair por volta da oitava época e passa a oscilar, subindo e descendo conforme os passos grandes jogam a rede para fora do mínimo. No `StepLR`, o efeito do degrau é imediato: na época em que a taxa cai de 2.0 para 0.2, a perda cai pela metade de uma época para a outra e a oscilação desaparece; a segunda queda, para 0.02, muda pouco, porque a essa altura já sobrou pouco a ajustar. Esse desenho em degraus é a assinatura do `StepLR` nas curvas de treinamento. O `CosineAnnealingLR` chega ao mesmo lugar sem descontinuidades, termina com a menor perda das três, e é a escolha usual quando o número de épocas é conhecido de antemão.

In [ ]:
histories["ReLU + He + batch norm + StepLR, lr 2.0"] = scheduled["StepLR"]
histories["ReLU + He + batch norm + cosine, lr 2.0"] = scheduled["cosine"]
plot_history(histories)

Cada peça atuou em um momento diferente: a inicialização acerta a escala do sinal antes do primeiro passo, a batch normalization mantém a escala enquanto os pesos mudam, o gradient clipping protege contra um passo isolado grande demais, e o scheduler reduz o passo quando o que falta é só ajuste fino. Nenhuma delas muda o que a rede é capaz de representar; elas só garantem que o gradiente chegue a todas as camadas com um tamanho utilizável.

## Exercícios

### Exercício 1

A rede com sigmoid não treinou. Ligue a batch normalization nela, com `MLP(nn.Sigmoid, batch_norm=True)`, e treine de novo com `lr=0.1`. Compare também as normas dos gradientes com e sem a normalização. Por que a camada resolve justamente o problema da sigmoid?

In [ ]:
# torch.manual_seed(42)
# sigmoid_bn_model = MLP(nn.Sigmoid, batch_norm=True).to(device)

### Exercício 2

Repita a inicialização com zeros usando uma constante diferente de zero, `nn.init.constant_(module.weight, 0.01)`. Agora todas as camadas recebem gradiente. Imprima as primeiras linhas do gradiente de uma camada oculta e explique por que a rede continua sem conseguir treinar.

In [ ]:
# def constant_init(module):
#     if isinstance(module, nn.Linear):
#         nn.init.constant_(module.weight, 0.01)

### Exercício 3

Troque o `CosineAnnealingLR` pelo `ReduceLROnPlateau`, que não segue fórmula fixa: ele observa uma métrica e multiplica a taxa por `factor` quando ela para de melhorar por `patience` épocas. Como a métrica aqui é a acurácia, ele precisa de `mode="max"`, e o `step` precisa receber a acurácia da época, o que exige uma pequena mudança em `train`. Em que época ele decidiu reduzir a taxa?

In [ ]:
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)